<a href="https://colab.research.google.com/github/hrbolek/learning/blob/master/notebooks/pydanticai.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# PydanticAI

## Cíl přednášky

Cílem není udělat z účastníků specialisty na neuronové sítě ani na vývoj velkých jazykových modelů.

Cílem je ukázat praktický pohled technika:

- co dnes znamená **LLM**,
- jak se z prostého volání LLM stane **agent**,
- jak agentovi zpřístupnit deterministické výpočty a data,
- jak získat typovaný a validovaný výstup,
- jak oddělit jazykovou interpretaci problému od klasického programu,
- jak se tyto principy vztahují k MCP a komplexnějším agentním systémům.

Praktická část je koncipována jako **hands-on workshop**. Většinu příkladů budeme spouštět přímo a sledovat, co agent skutečně dělá.

# Teorie kolem LLM

## Od expertních systémů k AI agentům

Myšlenka „programu, který řeší odborný problém pomocí znalostí a pravidel“ není nová.

Klasický expertní systém typicky obsahoval:

```text
uživatel
   ↓
inferenční mechanismus
   ↓
báze znalostí (expertní zkušenost) / báze dat (konkrétní případ)
   ↓
výsledek
```

Například:

```text
IF teplota > 90 °C
AND proud > jmenovitý proud
THEN upozorni na možné přetížení
```

Výhodou bylo, že chování systému bylo poměrně dobře vysvětlitelné a deterministické.

Nevýhodou bylo, že někdo musel:

- pravidla explicitně formulovat,
- znalosti převést do formální podoby,
- řešit velké množství výjimek,
- pravidla dlouhodobě udržovat.

## Co změnily velké jazykové modely

Velký jazykový model nepracuje primárně jako databáze pravidel.

Velmi zjednodušeně:

```text
text / tokeny
      ↓
 neuronová síť
      ↓
pravděpodobnost dalšího tokenu
      ↓
 generovaný výstup
```

To má zásadní praktickou výhodu:

> Vstup nemusíme přesně formalizovat.

Uživatel může napsat:

> „Máme motor 5 kW a zdá se mi, že bere moc proudu.“

LLM dokáže rozpoznat kontext, jazyk, pravděpodobný význam a potřebné informace.

Současně však LLM není automaticky:

- databáze aktuálních údajů,
- fyzikální simulátor,
- kalkulačka s garantovanou přesností,
- podnikový informační systém,
- autoritativní zdroj pravdy.

Proto je užitečné propojit LLM s klasickým softwarem.

## Agent jako spojení neurčitého a přesného světa

Pro praktické programování lze na agenta nahlížet takto:

```text
               přirozený jazyk
                     ↓
                    LLM
                     ↓
             rozhodnutí / plán
                     ↓
        ┌────────────┼────────────┐
        ↓            ↓            ↓
     funkce        data          API
        ↓            ↓            ↓
  přesný výpočet  realita    externí systém
        └────────────┼────────────┘
                     ↓
                    LLM
                     ↓
                  odpověď
```

LLM tedy nemusí nahrazovat existující software.

Může plnit roli vrstvy, která:

- porozumí neformálnímu zadání,
- zvolí vhodný nástroj,
- sestaví argumenty,
- interpretuje výsledek.

## Dnešní „LLM model“ často není jen neuronová síť

V běžné řeči dnes slovem **model** často označujeme celý systém, ke kterému přistupujeme přes API nebo chatovací rozhraní.

Je užitečné rozlišovat dvě vrstvy.

### 1. Základní jazykový model

```text
prompt
  ↓
neuronová síť
  ↓
generovaný text / tool call
```

Základní model má znalosti získané při trénování a následném dolaďování.

Sám o sobě ale nemusí mít přístup k aktuálním externím datům.

### 2. Současná LLM služba / agentní runtime

V praxi bývá model obalen dalšími schopnostmi:

```text
                    ┌── web search
                    ├── knowledge / RAG
                    ├── soubory
uživatel → LLM/API ─┼── code execution
                    ├── function tools
                    ├── MCP
                    └── další služby
```

Proto může dnešní systém působit, jako by „model znal internet, naše dokumenty nebo databázi“.

Přesnější formulace je:

> **Jazykový model má naučené znalosti; aplikace kolem něj mu může za běhu zpřístupňovat další znalosti a nástroje.**

Toto rozlišení bude důležité v celé praktické části.

Když později přidáme:

```python
tools=[mechanical_power]
```

nerozšiřujeme neuronovou síť ani ji znovu netrénujeme.

Pouze jí při konkrétním běhu zpřístupníme novou schopnost.

## Co je token

LLM nezpracovává přímo „slova“ ani „znaky“, ale **tokeny**.

Token může být například:

- celé krátké slovo,
- část delšího slova,
- interpunkce,
- část čísla,
- mezera spojená s následujícím textem.

Proto neplatí jednoduché pravidlo:

```text
1 token = 1 slovo
```

Pro praktické použití je důležité vědět, že tokeny určují zejména:

- velikost vstupu,
- velikost kontextového okna,
- množství generovaného výstupu,
- často také cenu volání modelu.

Čeština může mít proti angličtině odlišnou tokenizaci, protože záleží na konkrétním tokenizeru a slovníku modelu.

Pro velmi hrubý odhad lze u běžného českého textu používat:

| Text | Přibližný počet tokenů |
|---|---:|
| krátký odstavec | 50-150 |
| 1 strana A4 | ≈ 800 |
| 10 stran A4 | ≈ 8 000 |
| 100 stran A4 | ≈ 80 000 |
| 200 stran A4 | ≈ 160 000 |

Jiná praktická pomůcka:

**1 000 tokenů ≈ 1-2 A4 běžného textu.**

Jde pouze o řádový odhad. Výsledek silně ovlivňuje:

- jazyk,
- kód,
- tabulky,
- čísla,
- speciální znaky,
- konkrétní tokenizer modelu.

# LiteLLM na škole

Pro praktickou výuku nechceme měnit kód pokaždé, když použijeme jiný model.

Proto je užitečná architektura:

```text
notebook / aplikace
       ↓
OpenAI-compatible API
       ↓
    LiteLLM
       ↓
 ┌─────┼─────────────┐
 ↓     ↓             ↓
model A model B     model C
```

Aplikace tedy komunikuje s jednotným API.

Konkrétní model lze měnit konfigurací.

Na škole jsme v červnu 2026 zprovoznili LiteLLM, které umožňuje řízeně zpřístupnit jazykové modely, které máme pod smlouvami s Microsoftem.

## Modely dostupné v našem prostředí

Pro workshop lze pracovat například s modely:

| model | typické použití | poznámka |
|---|---|---|
| `gpt-5-nano` | jednoduché úlohy, extrakce, levné experimenty | vhodné pro většinu cvičení |
| `gpt-6-luna-short` | obecné úlohy | doplnit charakteristiku podle školního nasazení |
| `gpt-6-luna-long` | delší / náročnější úlohy | doplnit charakteristiku podle školního nasazení |


Pro představu lze uvažovat následující ceny za **1 milion textových tokenů**:

| Model / režim | Kontext | Vstup | Výstup |
|---|---:|---:|---:|
| `gpt-5-nano` | 400k | ~0.05€ / 1M | ~0.40€ / 1M |
| `gpt-6-luna-short` | krátký kontext | ~0.10€ / 1M | ~0.50€ / 1M |
| `gpt-6-luna-long` | dlouhý kontext | ~0.20€ / 1M | ~0.75€ / 1M |

`gpt-6-luna` má celkové kontextové okno přibližně **1.05 milionu tokenů** a maximální výstup 128k tokenů.


Důležitý je princip:

```text
větší prompt
+ delší historie
+ více tool výsledků
+ delší odpověď
= větší spotřeba tokenů
```

### Cena pro běžný dotaz

Předpokládejme:

```text
10 000 vstupních tokenů
 2 000 výstupních tokenů
```

Pro `gpt-6-luna-short`:

```text
vstup:
10 000 / 1 000 000 × $0.10 = $0.001

výstup:
2 000 / 1 000 000 × $0.50 = $0.001

celkem:
≈ $0.002
```

### Proč může být agent výrazně dražší než jednoduchý prompt?

U jednoduchého LLM volání si můžeme představit:

```text
uživatel
   ↓
jeden prompt
   ↓
jedna odpověď
```

Agent však může provést více kroků:

```text
uživatel
   ↓
LLM
   ↓
tool call
   ↓
LLM
   ↓
další tool call
   ↓
LLM
   ↓
další rozhodnutí
   ↓
LLM
   ↓
výsledná odpověď
```

Každé další volání modelu může znovu obsahovat velkou část dosavadního kontextu.

---

### Co se skutečně počítá do vstupu?

Do vstupních tokenů se typicky nepočítá pouze poslední věta uživatele.

Kontext agenta může obsahovat:

```text
instructions / system prompt
+
dotaz uživatele
+
historie konverzace
+
definice nástrojů
+
výsledky předchozích tool calls
+
data z knowledge / RAG
+
obsah dokumentů
+
mezivýsledky agentního řešení
```

Například:

```text
 2 000 tokenů instrukcí
 4 000 tokenů definic tools
20 000 tokenů historie
15 000 tokenů dokumentů z RAG
 3 000 tokenů výsledků předchozích tools
    50 tokenů nové otázky
----------------------------------------
≈44 000 vstupních tokenů
```

Uživatel přitom napsal jen krátkou otázku.

---

### Agentní smyčka náklady násobí

Představme si agenta, který při řešení jedné úlohy zavolá LLM pětkrát.

Pokud má v každém kroku přibližně 40 000 tokenů kontextu:

```text
1. volání    40 000 tokenů
2. volání    42 000 tokenů
3. volání    45 000 tokenů
4. volání    48 000 tokenů
5. volání    50 000 tokenů
-------------------------
celkem      225 000 vstupních tokenů
```

### Co si zapamatovat

1. **1 000 tokenů je řádově 1-2 A4 textu.**
2. Krátké LLM volání je dnes velmi levné.
3. Velký kontext může stále stát pouze jednotky až desítky centů.
4. Agent však může model zavolat několikrát během řešení jedné otázky.
5. Do kontextu patří i historie, tools, výsledky předchozích kroků apod.
6. Cena agentního systému proto závisí hlavně na:
   - velikosti kontextu,
   - počtu iterací,
   - množství připojených znalostí,
   - množství a velikosti tool výstupů.
7. Dobře navržený agent neposílá modelu všechno, co má k dispozici, ale pouze to, co je pro daný krok potřeba.

## Proč OpenAI-compatible rozhraní

V příkladech nebudeme model pevně zapisovat jako:

```python
Agent("openai:...")
```

Místo toho budeme mít na začátku notebooku:

```python
AI_BASE_URL
AI_API_KEY
AI_MODEL
```

a z nich vytvoříme objekt `MODEL`.

Výhoda:

```text
kód agenta
   ↓
zůstává stejný
   ↓
měníme endpoint nebo model konfigurací
```

To je důležité i pro reálné institucionální nasazení, kde aplikace nemusí komunikovat přímo s veřejným API výrobce modelu.

# Python

Pro dnešní workshop nepotřebujeme znát celý Python.

Budeme používat hlavně:

```python
variable = 10
```

```python
def function(x: float) -> float:
    return x * 2
```

```python
class Motor:
    ...
```

a dekorátory:

```python
@agent.tool_plain
def calculate(...):
    ...
```

Python je zde prostředek, kterým propojujeme LLM s klasickým programem.

## Python a PEP

Vývoj Pythonu je veřejně dokumentován prostřednictvím **PEP – Python Enhancement Proposals**.

PEP může popisovat například:

- nový jazykový prvek,
- změnu syntaxe,
- doporučený styl,
- proces vývoje Pythonu.

Známý příklad je:

```text
PEP 8 – Style Guide for Python Code
```

Důležitý princip pro technika:

> Python není jazyk, jehož vlastnosti vznikají nahodilým přidáváním funkcí do jedné implementace.

Významnější změny jsou diskutovány, formalizovány a dokumentovány.

Pro nás je dnes důležitá především velmi silná práce Python ekosystému s:

- type hints,
- datovými modely,
- asynchronním programováním,
- webovými API,
- AI knihovnami.

# Pydantic

Než se dostaneme k PydanticAI, je důležité pochopit základní myšlenku knihovny **Pydantic**.

Python dovoluje definovat datový model:

```python
from pydantic import BaseModel

class Motor(BaseModel):
    power_kw: float
    voltage_v: float
```

a následně zpracovávat data:

```python
motor = Motor(
    power_kw=7.5,
    voltage_v=400,
)
```

Výsledkem není pouze slovník.

Máme typovaný objekt s definovanou strukturou.

## Proč je Pydantic zajímavý ve světě LLM

LLM přirozeně generuje text.

Program však často potřebuje něco jiného:

```text
text LLM
   ↓
{
    "power_kw": 7.5,
    "voltage_v": 400
}
   ↓
validace
   ↓
Motor(...)
```

Tedy:

> LLM může být nedeterministické, ale hranice mezi LLM a naším programem může být typovaná a validovaná.

To je velmi důležitá myšlenka PydanticAI.

# PydanticAI

PydanticAI je Python framework / SDK pro tvorbu aplikací založených na LLM a agentech.

Pro náš workshop nás budou zajímat zejména:

```text
Agent
 ├── model
 ├── instructions
 ├── output type
 ├── tools
 ├── dependencies / RunContext
 └── message history
```

Současná dokumentace PydanticAI staví na typovaném agentním loopu a podporuje různé modelové providery i různá rozhraní pro běh stejného agenta, včetně terminálu.

## Minimální agent

Základní koncept je velmi malý:

```python
agent = Agent(
    MODEL,
    instructions="Jsi technický asistent."
)
```

Potom:

```python
result = agent.run_sync(
    "Jaký výkon potřebuji..."
)
```

V průběhu workshopu tento jednoduchý objekt budeme postupně rozšiřovat.

Nejdříve:

```text
Agent + LLM
```

potom:

```text
Agent + LLM + tools
```

a nakonec:

```text
Agent + LLM + tools + aplikační kontext + externí systémy
```

## Jak budeme v notebooku přidávat nástroje

Nástroje budeme definovat jako obyčejné Python funkce a předávat je při konstrukci agenta:

```python
def mechanical_power(force_n: float, velocity_m_s: float) -> float:
    return force_n * velocity_m_s


agent = Agent(
    MODEL,
    tools=[mechanical_power],
)
```

Tento způsob dobře odděluje:

```text
funkce / schopnosti
        ↓
    tools=[...]
        ↓
       Agent
```

Stejnou funkci tak lze snadno:

- otestovat bez LLM,
- použít ve více agentech,
- seskupovat s dalšími nástroji.

PydanticAI dovoluje v `tools=[...]` použít přímo funkce nebo explicitní objekty `Tool(...)`.
Explicitní `Tool` je vhodný tehdy, když potřebujeme jemněji řídit název, popis nebo další vlastnosti nástroje.

# MCP servery

## Proč MCP vzniklo

Pokud má každý agent vlastní způsob připojení ke každé službě, rychle vzniká velké množství jednorázových integrací:

```text
Agent A ── vlastní integrace ── GitHub
Agent A ── vlastní integrace ── databáze
Agent B ── jiná integrace ───── GitHub
Agent B ── jiná integrace ───── databáze
```

Model Context Protocol (MCP) vznikl jako otevřený standard pro standardizované propojení AI aplikací s nástroji a datovými zdroji.

Anthropic MCP veřejně představil 25. listopadu 2024.

## Základní architektura MCP

Bez MCP:

```text
Agent ──► speciálně napsaná Python funkce ──► systém
```

S MCP:

```text
Agent
  ↓
MCP client
  ↓
MCP server
  ↓
systém / nástroje / data
```

Analogii lze chápat podobně jako standardizovaný konektor:

> Agent nemusí znát interní implementaci služby. Potřebuje znát kontrakt, který služba zpřístupní.

MCP dnes podporují různé agentní frameworky a nástroje; PydanticAI má podporu MCP klientů i serverů.

## MCP a tools nejsou dvě různé myšlenky

Pro agenta je důležité, že má k dispozici schopnost:

```text
calculate_power(...)
```

Ta může být implementována:

```text
lokální Python funkcí
```

nebo:

```text
vzdáleným MCP serverem
```

Z hlediska didaktiky proto nejdříve pochopíme **tool calling**.

MCP přijde až poté jako způsob, jak tyto schopnosti standardizovat a sdílet.

# Praxe

Od této chvíle budeme principy z předchozí části skutečně spouštět.

Budeme postupovat po malých krocích:

```text
1. konfigurace LLM
2. první agent
3. instrukce
4. strukturovaný výstup
5. první tool
6. několik tools
7. vícekrokový problém
8. dependencies / RunContext
9. historie konverzace
10. externí systém
11. MCP jako zobecnění
```

Cílem není pouze získat správnou odpověď.

Budeme sledovat:

- co udělalo LLM,
- co udělal Python,
- kdy byl zavolán nástroj,
- odkud pocházejí data,
- co je deterministické a co ne.

## 0. Konfigurace prostředí

Všechny příklady používají stejný model a stejný OpenAI-compatible endpoint.

Díky tomu se dá notebook použít s:

- OpenAI,
- LiteLLM,
- institucionální LLM proxy,
- jiným OpenAI-compatible API.

Doporučení: přístupový klíč nastavte jako proměnnou prostředí.

In [ ]:
import os

AI_BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:4000/v1")

AI_API_KEY = os.getenv( "AI_API_KEY", "demo-key")

AI_MODEL = os.getenv("AI_MODEL", "gpt-5-nano")

VERBOSE_TOOLS = True

In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=AI_BASE_URL,
    api_key=AI_API_KEY,
)

MODEL = OpenAIModel(
    AI_MODEL,
    provider=provider,
)

print("LLM endpoint:", AI_BASE_URL)
print("Model:", AI_MODEL)

### Pomocná funkce pro výuku

Při workshopu je užitečné vidět, kdy agent skutečně zavolal Python funkci.

Proto budeme nástroje vypisovat do terminálu pomocí prefixu `[TOOL]`.

In [ ]:
def tool_log(message: str) -> None:
    if VERBOSE_TOOLS:
        print(f"[TOOL] {message}")

In [ ]:
def test_agent(agent, prompt: str, **run_kwargs):
    """
    Spustí jeden testovací dotaz a přehledně vypíše výsledek.

    Použití:
        test_agent(agent, "Jaký výkon ...?")
        test_agent(agent, "...", deps=lab)
    """
    print("=" * 72)
    print("PROMPT:")
    print(prompt)
    print("-" * 72)

    result = agent.run_sync(
        prompt,
        **run_kwargs,
    )

    print("OUTPUT:")
    print(result.output)
    print("=" * 72)

    return result

## Interaktivní test přímo v notebooku

Notebook je zároveň prostředí pro spuštění.

Pro jednorázové testy budeme používat:

```python
test_agent(agent, "dotaz")
```

Pro krátkou interaktivní konverzaci lze spustit následující funkci:

```python
chat(agent)
```

Ukončení: `exit` nebo `quit`.

Na rozdíl od externího CLI zůstává celý experiment přímo v notebooku.

In [ ]:
def chat(agent, *, deps=None):
    history = []

    while True:
        prompt = input(">>> ").strip()

        if prompt.lower() in {"exit", "quit"}:
            break

        run_kwargs = {
            "message_history": history,
        }

        if deps is not None:
            run_kwargs["deps"] = deps

        result = agent.run_sync(
            prompt,
            **run_kwargs,
        )

        print(result.output)
        history = result.all_messages()

# 1. První agent

Nejdříve vytvoříme agenta bez dalších nástrojů.

Agent má:

- model,
- instrukce,
- vstup uživatele,
- textovou odpověď.

Vyzkoušejte například přímo v následující kódové buňce:

> Jaký výkon je potřeba pro zvedání břemene 250 kg rychlostí 0.3 m/s?

V tomto materiálu používáme především `run_sync()`, protože notebook je zároveň naším spouštěcím a testovacím prostředím.

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent pro strojaře a elektrotechniky.
    Odpovídej stručně, technicky a používej SI jednotky.
    """,
)

agent

In [ ]:
# Jednorázový test přímo v notebooku:
test_agent(
    agent,
    "Jaký výkon je potřeba pro zvedání břemene 250 kg rychlostí 0.3 m/s?",
)

# Volitelně lze spustit interaktivní konverzaci:
# chat(agent)

In [ ]:
result = agent.run_sync(
    "Jaký výkon je potřeba pro zvedání břemene 250 kg rychlostí 0.3 m/s?"
)

print(result.output)

## Co se právě stalo?

Zjednodušeně:

```text
uživatel
   ↓
Agent
   ↓
LLM
   ↓
odpověď
```

LLM zatím nemá žádný specializovaný nástroj. Výpočet si provádí samo.

# 2. Instrukce mění chování agenta

Model zůstává stejný.

Měníme pouze instrukce.

In [ ]:
agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent.

    Při řešení:
    - používej SI jednotky,
    - napiš použitý fyzikální vztah,
    - dosaď hodnoty,
    - uveď výsledek,
    - upozorni na chybějící údaje.
    """,
)

In [ ]:
result = agent.run_sync(
    "Jaký výkon je potřeba pro zvedání břemene 250 kg rychlostí 0.3 m/s?"
)

print(result.output)

### Experiment

Změňte instrukce například tak, aby agent:

- odpovídal pouze jednou větou,
- vysvětloval řešení studentovi prvního ročníku,
- vracel pouze výsledek a jednotku.

Důležitý poznatek:

> Stejný LLM může být součástí různě se chovajících agentů.

# 3. Strukturovaný výstup

LLM nemusí vracet pouze text.

Pomocí Pydantic modelu můžeme požadovat přesně definovanou strukturu dat.

In [ ]:
from pydantic import BaseModel


class MotorSpecification(BaseModel):
    power_kw: float
    voltage_v: float
    rpm: float
    efficiency: float | None = None

In [ ]:
agent = Agent(
    MODEL,
    output_type=MotorSpecification,
)

result = agent.run_sync("""
Motor má výkon 7.5 kW,
napájecí napětí 400 V,
otáčky 1450 min-1
a účinnost 91 %.
""")

print(result.output)
print(type(result.output))

Výstup není obyčejný text.

Je to validovaný Python objekt:

```text
LLM
 ↓
Pydantic validace
 ↓
Python objekt
```

To je jeden z klíčových kroků od „chatbota“ k běžné softwarové komponentě.

# 4. První nástroj

Nyní dáme agentovi Python funkci, kterou může použít.

LLM nebude výkon počítat samo. Rozhodne, zda a jak má zavolat nástroj.

In [ ]:
def mechanical_power(
    force_n: float,
    velocity_m_s: float,
) -> float:
    """
    Calculate mechanical power.

    Args:
        force_n: Force in newtons.
        velocity_m_s: Velocity in metres per second.

    Returns:
        Mechanical power in watts.
    """
    result = force_n * velocity_m_s

    tool_log(
        f"mechanical_power(force_n={force_n}, "
        f"velocity_m_s={velocity_m_s}) -> {result}"
    )

    return result


# Funkci lze otestovat i bez LLM.
assert mechanical_power(3500, 1.2) == 4200


agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent.

    Pokud lze výpočet provést dostupným nástrojem,
    použij jej.
    """,
    tools=[
        mechanical_power,
    ],
)

In [ ]:
result = test_agent(
    agent,
    "Jaký výkon je potřeba při síle 3500 N a rychlosti 1.2 m/s?",
)

### Co se stalo?

Agentní smyčka vypadá zjednodušeně takto:

```text
uživatel
   ↓
  LLM
   ↓
rozhodnutí použít tool
   ↓
mechanical_power(...)
   ↓
Python
   ↓
výsledek toolu
   ↓
  LLM
   ↓
odpověď
```

LLM nevykonává Python kód.

LLM pouze rozhoduje, **který nástroj použít a s jakými argumenty**.

# 5. Několik technických nástrojů

Rozšíříme agenta o několik funkcí.

Agent si sám vybere, které z nich potřebuje.

In [ ]:
import math


def resistance_force(
    mass_kg: float,
    resistance_coefficient: float,
) -> float:
    """
    Calculate resistance force for a simplified conveyor model.
    """
    result = mass_kg * 9.81 * resistance_coefficient

    tool_log(
        f"resistance_force(mass_kg={mass_kg}, "
        f"resistance_coefficient={resistance_coefficient}) "
        f"-> {result:.2f} N"
    )

    return result


def mechanical_power(
    force_n: float,
    velocity_m_s: float,
) -> float:
    """
    Calculate mechanical power in watts.
    """
    result = force_n * velocity_m_s

    tool_log(
        f"mechanical_power(force_n={force_n}, "
        f"velocity_m_s={velocity_m_s}) "
        f"-> {result:.2f} W"
    )

    return result


def three_phase_current(
    power_w: float,
    voltage_v: float,
    power_factor: float,
    efficiency: float = 1.0,
) -> float:
    """
    Calculate approximate line current of a three-phase load.
    """
    result = power_w / (
        math.sqrt(3)
        * voltage_v
        * power_factor
        * efficiency
    )

    tool_log(
        f"three_phase_current(power_w={power_w}, "
        f"voltage_v={voltage_v}, "
        f"power_factor={power_factor}, "
        f"efficiency={efficiency}) "
        f"-> {result:.2f} A"
    )

    return result


# Nejprve lze nástroje otestovat jako obyčejné Python funkce.
assert round(resistance_force(1800, 0.08), 2) == 1412.64
assert mechanical_power(1000, 2) == 2000
assert round(three_phase_current(11000, 400, 0.84, 0.91), 2) > 0


TECHNICAL_TOOLS = [
    resistance_force,
    mechanical_power,
    three_phase_current,
]


agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent pro strojaře a elektrotechniky.

    Pro výpočty používej dostupné nástroje.
    Pokud některý údaj chybí, raději se zeptej nebo na něj upozorni.
    """,
    tools=TECHNICAL_TOOLS,
)

In [ ]:
result = test_agent(
    agent,
    """
    Máme dopravník.

    Pohybující se hmotnost je 1800 kg.
    Součinitel odporu je 0.08.
    Rychlost pásu je 1.5 m/s.

    Motor je třífázový 400 V,
    účinnost je 90 %
    a cos phi = 0.85.

    Urči odporovou sílu, potřebný mechanický výkon
    a proud motoru.
    """,
)

### Diskuse

Agent může potřebovat několik tool callů za sebou.

Například:

```text
resistance_force(...)
        ↓
     síla
        ↓
mechanical_power(...)
        ↓
     výkon
        ↓
three_phase_current(...)
        ↓
      proud
```

Tím vzniká jednoduché vícekrokové agentní řešení.

# 6. Kdy tool použít a kdy ne?

Vyzkoušejte dvě zcela odlišné otázky.

In [ ]:
questions = [
    "Kolik je 127 krát 382?",
    """
    Jaký proud přibližně odebírá třífázový motor
    s elektrickým příkonem 11 kW,
    napětím 400 V, účiníkem 0.84 a účinností 0.91?
    """,
]

for question in questions:
    print("\nQUESTION:")
    print(question)

    result = agent.run_sync(question)

    print("\nANSWER:")
    print(result.output)

Důležitá otázka pro workshop:

> Jak agent poznal, že má nástroj použít?

V kódu nemáme například:

```python
if "motor" in prompt:
    three_phase_current(...)
```

Rozhodnutí je součástí práce LLM.

# 7. Agent není fyzikální model

Agent může dostat nedostatečně zadaný problém.

Vyzkoušejte například:

In [ ]:
result = agent.run_sync(
    "Mám motor 5 kW. Jaký proud bude odebírat?"
)

print(result.output)

Pro korektní výpočet mohou chybět například:

- napětí,
- počet fází,
- účinnost,
- účiník.

Agent by neměl bezdůvodně vymýšlet hodnoty.

To je důležitá hranice mezi:

- jazykovým modelem,
- odborným modelem,
- měřením,
- deterministickým výpočtem.

# 8. Dependencies a RunContext

Nástroje nemusí pracovat pouze se vstupními argumenty.

Často potřebují přístup ke kontextu aplikace:

- databázi,
- přihlášenému uživateli,
- měřicímu systému,
- konfiguraci,
- API klientovi.

V PydanticAI se tento kontext předává pomocí dependencies.

In [ ]:
from dataclasses import dataclass
from pydantic_ai import RunContext


@dataclass
class Laboratory:
    temperature_c: float
    voltage_v: float
    current_a: float

In [ ]:
def read_temperature(ctx: RunContext[Laboratory]) -> float:
    """Read current laboratory temperature."""
    tool_log("read_temperature()")
    return ctx.deps.temperature_c


def read_voltage(ctx: RunContext[Laboratory]) -> float:
    """Read current measured voltage."""
    tool_log("read_voltage()")
    return ctx.deps.voltage_v


def read_current(ctx: RunContext[Laboratory]) -> float:
    """Read current measured current."""
    tool_log("read_current()")
    return ctx.deps.current_a


LAB_TOOLS = [
    read_temperature,
    read_voltage,
    read_current,
]


lab_agent = Agent(
    MODEL,
    deps_type=Laboratory,
    instructions="""
    Jsi asistent laboratorního pracoviště.
    Pokud potřebuješ aktuální měření, použij nástroje.
    """,
    tools=LAB_TOOLS,
)

In [ ]:
result = test_agent(
    lab_agent,
    "Jaké je aktuální napětí a teplota v laboratoři?",
    deps=lab,
)

Schéma:

```text
                  Agent
                    │
        ┌───────────┼────────────┐
        │           │            │
        ▼           ▼            ▼
       LLM       instructions    deps
                                  │
                              Laboratory
                                  │
                                  ▼
                                tool
```

V reálné aplikaci mohou dependencies obsahovat například:

```text
deps
 ├── database
 ├── GraphQL client
 ├── REST client
 ├── logged user
 ├── services
 └── configuration
```

# 9. Historie konverzace

Jednotlivé `run_sync()` požadavky jsou samostatné, pokud historii explicitně nepředáme.

Historii konverzace lze uchovávat a znovu posílat agentovi.

In [ ]:
chat_agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent.
    Pamatuj si parametry technického problému z předchozí konverzace.
    """,
)

history = []

In [ ]:
result = chat_agent.run_sync(
    "Navrhujeme pohon dopravníku o výkonu 4 kW.",
    message_history=history,
)

print(result.output)

history = result.all_messages()

In [ ]:
result = chat_agent.run_sync(
    "Použij napětí 400 V a účinnost 88 %.",
    message_history=history,
)

print(result.output)

history = result.all_messages()

In [ ]:
result = chat_agent.run_sync(
    "Co už o tomto pohonu víš?",
    message_history=history,
)

print(result.output)

history = result.all_messages()

### Jednoduchá terminálová smyčka

Následující buňku lze použít při workshopu.

Ukončení: `exit` nebo `quit`.

In [ ]:
# history = []
#
# while True:
#     prompt = input(">>> ")
#
#     if prompt.lower() in {"exit", "quit"}:
#         break
#
#     result = chat_agent.run_sync(
#         prompt,
#         message_history=history,
#     )
#
#     print(result.output)
#
#     history = result.all_messages()

# 10. Agent nad jednoduchým technickým systémem

Dosud jsme používali výpočty a simulované laboratorní hodnoty.

Nyní budeme předstírat, že máme jednoduché API nebo databázi se stavem strojů.

In [ ]:
machines = {
    "M1": {
        "temperature_c": 63.2,
        "rpm": 1482,
        "current_a": 8.3,
    },
    "M2": {
        "temperature_c": 91.7,
        "rpm": 1390,
        "current_a": 13.4,
    },
    "M3": {
        "temperature_c": 58.4,
        "rpm": 1491,
        "current_a": 7.9,
    },
}

In [ ]:
def get_machine_state(machine_id: str) -> dict:
    """Get current telemetry for a machine."""
    tool_log(f"get_machine_state({machine_id})")

    machine_id = machine_id.upper()

    if machine_id not in machines:
        raise ValueError(f"Unknown machine: {machine_id}")

    return machines[machine_id]


# Test funkce bez LLM:
assert get_machine_state("M1")["rpm"] == 1482


machine_agent = Agent(
    MODEL,
    instructions="""
    Jsi technický asistent údržby.

    Máš přístup ke stavům strojů.
    Pro zjištění aktuálních údajů vždy použij dostupný nástroj.

    Neprováděj kategorické diagnostické závěry bez dostatečných dat.
    Popiš pouze pozorované odchylky a možné důvody.
    """,
    tools=[
        get_machine_state,
    ],
)

In [ ]:
result = test_agent(
    machine_agent,
    "Porovnej stav motorů M1 a M2. "
    "Který vykazuje výraznější odchylky a v čem?",
)

Tohle je už velmi blízko skutečnému použití:

```text
LLM
 │
 Agent
 │
 ├── výpočty
 ├── databáze
 ├── měřicí systém
 ├── REST API
 ├── GraphQL API
 └── podnikový informační systém
```

# 11. Klasický program vs. agent

U přesně definovaného problému je klasická funkce často nejlepší řešení.

In [ ]:
def conductor_resistance(
    resistivity_ohm_mm2_m: float,
    length_m: float,
    area_mm2: float,
) -> float:
    return resistivity_ohm_mm2_m * length_m / area_mm2


print(
    conductor_resistance(
        resistivity_ohm_mm2_m=0.0175,
        length_m=35,
        area_mm2=2.5,
    )
)

Agent je užitečný tam, kde uživatel zadává problém přirozeným jazykem:

```text
"Máme 35 metrů měděného vodiče 2.5 mm².
Jaký bude mít odpor?"
```

Rozdělení rolí:

```text
neurčitý problém / přirozený jazyk
              ↓
             LLM
              ↓
      výběr nástroje
              ↓
 přesný deterministický výpočet
```

Praktická zásada:

> LLM používejme tam, kde je problém neurčitý.  
> Klasický program používejme tam, kde je problém přesně definovaný.

# 12. MCP – základní princip

Dosud jsme nástroje implementovali přímo v Pythonu:

```text
Agent ──► Python function
```

U větších systémů může být výhodné nástroje zpřístupnit standardizovaně:

```text
Agent ──► MCP client ──► MCP server ──► zařízení / služba / data
```

MCP tedy není agent.

Je to standardizovaný způsob, jak agentovi nabídnout schopnosti a zdroje.

Pro tento workshop stačí porozumět principu. Praktické nasazení MCP může být samostatné pokračování.

# 13. Shrnutí

Postupně jsme sestavili stále schopnější systém:

```text
LLM
```

↓

```text
Agent
+ instructions
```

↓

```text
Agent
+ structured output
```

↓

```text
Agent
+ Python tools
```

↓

```text
Agent
+ application context
```

↓

```text
Agent
+ external systems
```

↓

```text
Agent
+ MCP
```

Základní agentní princip lze shrnout:

```text
Agent = LLM + instrukce + nástroje + kontext + agentní smyčka
```

## Kam se lze dostat dál?

Komplexnější agent může mít k dispozici schopnosti jako:

- čtení souborů,
- vyhledávání v projektu,
- spouštění příkazů,
- práci s databází,
- API,
- další agenty,
- vzdálené služby přes MCP.

Příkladem takového systému může být **Coder**.

Pro dnešní workshop ale není potřeba jej prakticky implementovat.

Důležité je, že princip zůstává stejný:

```text
uživatel
   ↓
  LLM
   ↓
rozhodnutí
   ↓
nástroj / služba / výpočet
   ↓
výsledek
   ↓
  LLM
   ↓
odpověď
```

# Doporučené úkoly pro účastníky

1. Změňte instrukce prvního agenta.
2. Přidejte nový parametr do `MotorSpecification`.
3. Napište vlastní tool pro výpočet momentu:
   \[
   M = \frac{P}{\omega}
   \]
4. Přidejte tool pro synchronní otáčky elektromotoru.
5. Nechte agenta vyřešit problém, který vyžaduje dva nebo tři tool calls.
6. Rozšiřte `Laboratory` o další měřenou veličinu.
7. Přidejte do `machines` další stroj a nechte agenta porovnat všechny stroje.
8. Zkuste formulovat neúplné zadání a sledujte, zda agent upozorní na chybějící údaje.